# Bài 3 · NumPy và tư duy vector hoá

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Viện TTNT, UET-VNU**

> 💡 File → **Save a copy in Drive** trước khi sửa.

**Mục tiêu bài học** — sau notebook này, bạn sẽ:

1. Giải thích cách NumPy lưu mảng số và vì sao phép toán trên mảng thường nhanh hơn vòng `for` Python.
2. Dùng `shape`, `dtype`, `strides` để đọc cấu trúc mảng; phân biệt view với bản sao.
3. Viết phép tính theo hàng, cột bằng broadcasting và kiểm tra kết quả trên ví dụ nhỏ.
4. Diễn giải trung bình, trung vị, độ lệch chuẩn và kết quả lấy mẫu trong bài tập lớn.

Dự đoán trước khi chạy; làm bài trong ô trống rồi đối chiếu ô tham khảo. Dữ liệu trong notebook được tự tạo để tính tay.


## 1. Cấu trúc của mảng NumPy

Bắt đầu từ cách biểu diễn dữ liệu trong bộ nhớ.


### Mảng hai chiều

**ndarray** là kiểu mảng của NumPy. Mảng A dưới đây có 4 hàng, 3 cột.

- `A.shape`: kích thước từng chiều, ở đây là `(4, 3)`.
- `A.ndim`: số chiều, ở đây là 2.
- `A.dtype`: kiểu phần tử, ở đây là `int64`, chiếm 8 byte.

Các ví dụ sửa dữ liệu dùng một bản sao của A.


In [ ]:
import numpy as np
A = np.array([[10, 12, 11],
              [20, 21, 24],
              [30, 33, 31],
              [40, 44, 42]], dtype=np.int64)
print("shape:", A.shape, "| số chiều:", A.ndim, "| dtype:", A.dtype)


### List và ndarray lưu gì?

Với CPython, list là dãy tham chiếu tới đối tượng. ndarray có dtype số lưu giá trị trong các ô cùng kích thước. Dãy tham chiếu của list cũng liền nhau; sự khác biệt là nội dung của mỗi ô. Một đối tượng có thể được nhiều tham chiếu dùng chung.


### Mảng tổ chức dữ liệu như thế nào?

**Metadata** là thông tin mô tả mảng. A có `shape = (4, 3)` nhưng vùng dữ liệu chứa 12 giá trị xếp liên tiếp. `dtype` cho biết số byte của mỗi phần tử và cách đọc các byte ấy thành một giá trị. A mới tạo được lưu theo hàng: `10, 12, 11, 20, 21, 24, …`. Cách sắp xếp này gọi là C-order.

Ở phần 3, ta sẽ dùng `strides` để mô tả các bước dịch chuyển giữa các phần tử.


### Tính dung lượng từ cách lưu mảng

- `A.itemsize`: số byte của **một phần tử**.
- `A.nbytes`: số byte của **tất cả phần tử**.

Đây là thuộc tính: viết `A.itemsize`, không thêm `()`. Với A, 4 × 3 phần tử × 8 byte = 96 byte dữ liệu, chưa gồm metadata.

Đổi shape từ `(4,3)` sang `(2,6)` có làm giảm dung lượng dữ liệu không? Giải thích từ số phần tử và số byte mỗi phần tử.


In [ ]:
print("Mỗi phần tử:", A.itemsize, "byte")
print("Phần dữ liệu của A:", A.nbytes, "byte")


### Vì sao mảng số tốn ít bộ nhớ?

Cả list và ndarray dưới đây cùng lưu dãy **1000, 1001, …, 10999**. List cần chỗ cho bảng tham chiếu và các đối tượng `int`; ndarray có dtype `int64` lưu trực tiếp các giá trị số.

`sys.getsizeof(items_do)` tính đối tượng list và vùng tham chiếu. Ta cộng thêm kích thước các đối tượng `int` mà list trỏ tới. Trong ví dụ này, các giá trị khác nhau nên không đếm một đối tượng hai lần. Mảng NumPy sở hữu vùng dữ liệu nên `sys.getsizeof(arr_do)` tính cả vùng dữ liệu và metadata. Đây là kích thước đối tượng trên máy chạy, không phải RAM của toàn tiến trình.


In [ ]:
import sys
import platform

items_do = list(range(1000, 11_000))  # dãy 1000, 1001, ..., 10999; mỗi giá trị chỉ xuất hiện một lần
arr_do = np.array(items_do, dtype=np.int64)  # mảng sở hữu vùng dữ liệu
byte_refs = sys.getsizeof(items_do)
byte_ints = sum(sys.getsizeof(v) for v in items_do)
print("Python", platform.python_version(), "| NumPy", np.__version__)
print("List: tham chiếu + đối tượng int =", byte_refs, "+", byte_ints, "byte")
print("ndarray: dữ liệu / toàn đối tượng =", arr_do.nbytes, "/", sys.getsizeof(arr_do), "byte")
# Không phải phép đo RAM toàn bộ tiến trình; không dùng sum(getsizeof) nếu có tham chiếu dùng chung.


### Khi kết quả vượt giới hạn kiểu số

`int8` dùng 1 byte, chỉ lưu số nguyên từ −128 đến 127. Dự đoán `127 + 1` trong `int8`, rồi chạy. So với `int` Python, vốn có thể tăng số byte để lưu số nguyên lớn.

`astype` tạo mảng với dtype mới. Nếu cần kiểu lớn hơn, phải đổi **trước khi tính**; đổi kiểu sau khi tràn không khôi phục kết quả đúng.


In [ ]:
x8 = np.array([127], dtype=np.int8)
print("int Python:", 127 + 1)
print("Tính trong int8:", x8 + 1)
print("Đổi sang int16 trước:", x8.astype(np.int16) + 1)
assert (x8 + 1)[0] == -128
assert (x8.astype(np.int16) + 1)[0] == 128


### Số chữ số in ra và độ chính xác

Hai việc khác nhau:

- **Dtype** quyết định giá trị gần đúng được lưu. float32 dùng 4 byte; float64 dùng 8 byte và có độ chính xác cao hơn.
- **Định dạng in** quyết định hiển thị bao nhiêu chữ số. `f"{gia_tri:.12f}"` yêu cầu 12 chữ số sau dấu chấm cho cả hai kiểu.

Với 1/3, giá trị float32 được lưu bằng chính xác `11184811/33554432`, tức `0.3333333432674407958984375`. Đây là số gần 1/3 mà float32 biểu diễn được. Các chữ số khác 3 không ngẫu nhiên; chúng thuộc về số gần đúng ấy.

float64 gần 1/3 hơn, nên in 12 chữ số vẫn thấy toàn chữ số 3. Điều này không có nghĩa float64 lưu chính xác 1/3. In nhiều chữ số không làm tăng độ chính xác đã lưu.


In [ ]:
thuong32 = np.float32(1) / np.float32(3)
thuong64 = np.float64(1) / np.float64(3)
print(f"float32, in 12 số: {thuong32:.12f}")
print(f"float64, in 12 số: {thuong64:.12f}")
print(f"Cùng float32, chỉ in 6 số: {thuong32:.6f}")
assert f"{thuong32:.12f}" == "0.333333343267"
assert f"{thuong64:.12f}" == "0.333333333333"
assert f"{thuong32:.6f}" == "0.333333"
assert float(thuong32).as_integer_ratio() == (11184811, 33554432)


### Bài tập 1: đổi kiểu lúc nào?

Cần nhân đôi các số trong `x`. Hai cách sau có cho cùng kết quả?

```python
x = np.array([50, 100, 120], dtype=np.int8)
a = (x * 2).astype(np.int16)
b = x.astype(np.int16) * 2
```

1. Dự đoán giá trị của `a` và `b`. Cách nào đúng?
2. Tính `nbytes` của `x`, `a`, `b`.
3. Vì sao đổi sang `int16` vẫn có thể cho kết quả sai?


In [ ]:
# Viết dự đoán và lời giải của bạn ở đây.


**Đối chiếu sau khi đã làm bài.** Chạy ô tham khảo rồi giải thích mọi kết quả khác dự đoán.


In [ ]:
x_bt = np.array([50, 100, 120], dtype=np.int8)
a_bt = (x_bt * 2).astype(np.int16)
b_bt = x_bt.astype(np.int16) * 2
for ten, mang in [("x", x_bt), ("a", a_bt), ("b", b_bt)]:
    print(ten, mang, mang.dtype, mang.nbytes, "byte")
# Đổi kiểu sau phép nhân không khôi phục được giá trị đã tràn.


### So sánh số thực cần tính đến sai số

`np.isclose` kiểm tra hai giá trị có gần nhau trong sai số cho phép. Ở ví dụ này, `rtol=0, atol=1e-12` nghĩa là chấp nhận chênh lệch tuyệt đối không quá 10⁻¹². Chọn mức sai số theo bài toán.


In [ ]:
a = np.array([0.1, 0.2])
print(a.sum())
print(a.sum() == 0.3)
print(np.isclose(a.sum(), 0.3, rtol=0, atol=1e-12))
assert a.sum() != 0.3
assert np.isclose(a.sum(), 0.3, rtol=0, atol=1e-12)


## 2. Phép tính được thực hiện ở đâu?

Cấu trúc mảng chỉ phát huy tác dụng khi cách tính tận dụng được nó.


### Nhân đôi các giá trị

Hai cách cùng nhân từng giá trị với 2. Vòng lặp được thực hiện ở đâu trong mỗi cách?


In [ ]:
xs = [10, 12, 11]
x = np.array(xs, dtype=np.int64)
print([value * 2 for value in xs])
print(x * 2)


### Vòng lặp Python xử lý đối tượng

Vòng lặp trên list xử lý đối tượng Python tại mỗi lượt. Khi lấy từng phần tử của ndarray qua vòng for Python, NumPy còn phải biểu diễn giá trị đó thành đối tượng scalar để Python sử dụng. Chỉ thay vùng chứa chưa loại bỏ chi phí vòng lặp.


### Phép nhân trên mảng và ufunc

**ufunc** (*universal function*) là hàm NumPy áp dụng một phép toán lên từng phần tử. Với ndarray, `x * 2` gọi phép nhân `np.multiply`. Mỗi ô đầu ra nhận giá trị của ô đầu vào nhân với 2.


In [ ]:
print(np.multiply(x, 2))
np.testing.assert_array_equal(x * 2, np.multiply(x, 2))


### Thông dịch và biên dịch

- **CPython:** mã Python được biên dịch thành bytecode; trình thông dịch thực hiện bytecode. Vòng `for` trên list còn phải lấy các đối tượng số qua tham chiếu.
- **NumPy:** vòng lặp số học viết bằng C đã được biên dịch thành mã máy khi xây dựng thư viện. Khi gọi `x * 2`, NumPy chọn mã phù hợp dtype rồi xử lý các ô số của mảng.

Vẫn có vòng lặp. Phần giảm đi là việc xử lý lệnh và đối tượng Python trong từng lượt. Bytecode chưa phải mã máy; NumPy cũng không tự biên dịch vòng `for` của bạn.

Nguồn: [Python: bytecode](https://docs.python.org/3/glossary.html#term-bytecode), [NumPy: ufunc](https://numpy.org/doc/stable/user/basics.ufuncs.html).


### Dtype quyết định cách tính

NumPy có sẵn các đoạn **mã máy** xử lý từng kiểu số. Mã máy là mã đã biên dịch để CPU thực thi.

Trong `np.multiply(x, 2)`, nếu x có dtype `int64` thì ví dụ này dùng đoạn mã đọc và nhân số nguyên 8 byte. Đổi x sang `int32` thì dùng số nguyên 4 byte; đổi sang `float64` thì dùng số thực 8 byte. `int64` và `float64` cùng chiếm 8 byte nhưng diễn giải các bit và thực hiện phép tính khác nhau.

Ví dụ dưới đây giúp quan sát dtype vào/ra, không yêu cầu học API nội bộ. Khi trộn các kiểu khác nhau, còn phải xét quy tắc chuyển kiểu của NumPy.


In [ ]:
for dtype_vi_du in (np.int32, np.int64, np.float64):
    x_vi_du = np.array([10, 12, 11], dtype=dtype_vi_du)
    y_vi_du = np.multiply(x_vi_du, 2)
    print(x_vi_du.dtype, "->", y_vi_du.dtype, ":", y_vi_du)
    assert y_vi_du.dtype == x_vi_du.dtype


### Đọc, nhân, ghi trên vùng dữ liệu

Với x là mảng `int64` liên tục, `y = np.multiply(x, 2)` có thể hiểu qua các bước:

1. Đọc 8 byte tại `x[0]` thành số nguyên 10.
2. Tính phép nhân số nguyên 10 × 2 = 20.
3. Ghi 20 vào 8 byte tại `y[0]`.
4. Dịch 8 byte để tới `x[1]`, `y[1]`, rồi tiếp tục với 12 và 11.

Python gọi phép toán mảng một lần. Vòng lặp bên trong NumPy đọc và ghi trực tiếp các giá trị số, không cần một đối tượng Python cho mỗi kết quả.

Đây là mô hình để hiểu cách tính. Tối ưu SIMD có thể xử lý nhiều số trong một lệnh máy, tuỳ phép toán, bản NumPy và CPU. Vector hoá không mặc nhiên là chạy nhiều luồng.


### Đọc các ô liền nhau tận dụng cache

**Cache** là bộ nhớ đệm nhỏ, gần CPU và nhanh hơn RAM. Ví dụ khi nhân lần lượt 10, 12, 11, 20 với 2:

1. Nếu khối chứa 10 chưa có trong cache, CPU cần nạp khối từ RAM. Các giá trị liền kề như 12, 11, 20 cũng được nạp cùng.
2. Khi đọc tiếp 12, nếu khối vẫn còn trong cache, CPU dùng giá trị đã có sẵn thay vì nạp lại từ RAM.

Khi NumPy duyệt các ô liên tiếp, các giá trị cần đọc tiếp thường đã có trong cache, giúp giảm thời gian chờ dữ liệu. Đây là một lợi ích của cách bố trí và truy cập mảng số liên tục.

Khối 4 ô trong hình chỉ để minh hoạ, không phải kích thước cache line thực tế. Hiệu quả còn phụ thuộc cách duyệt, kích thước mảng và phép toán.


### SIMD: xử lý nhiều số cùng lúc

**SIMD** (Single Instruction, Multiple Data): một lệnh máy thực hiện cùng thao tác trên nhiều số. Một lõi CPU cũng có thể dùng SIMD.

Ví dụ nhân theo nhóm: `[10, 12, 11, 20] × 2 → [20, 24, 22, 40]`.

NumPy tự chọn mã phù hợp với CPU. Việc dùng SIMD còn tuỳ phép toán, dtype, độ dài và cách bố trí mảng.

Trong bản NumPy 2.5.2 trên máy đo của bài, phép nhân mảng int64 lớn, liên tục với 2 có nhánh xử lý bốn số mỗi nhóm bằng một chuỗi lệnh SIMD. Ví dụ chỉ có ba phần tử đi qua nhánh xử lý từng số. Đây là kết quả kiểm tra CPU dispatch và mã máy đã cài, không phải quy tắc cho mọi máy; cũng không có nghĩa bốn phép nhân int64 chỉ cần một lệnh.

Nguồn: [NumPy: SIMD](https://numpy.org/doc/stable/reference/simd/index.html).


### NumPy có chạy nhiều luồng không?

**Nhiều luồng**: chia công việc cho các luồng có thể chạy đồng thời trên nhiều lõi CPU.

- `x * 2` thường chạy một luồng, có thể dùng SIMD.
- Phép nhân ma trận số thực `A @ B` có thể dùng nhiều luồng qua BLAS (thư viện đại số tuyến tính, như OpenBLAS hoặc MKL), tuỳ bản cài và cấu hình.
- Nhiều luồng và SIMD có thể cùng được dùng.

NumPy vẫn có thể nhanh hơn vòng `for` Python khi chỉ chạy một luồng, nhờ mã đã biên dịch và cách lưu mảng số. Benchmark của bài không đo riêng mức tăng tốc do SIMD.

Nguồn: [NumPy: số luồng tính toán](https://numpy.org/doc/stable/reference/global_state.html#number-of-threads-used-for-linear-algebra).


### So sánh tốc độ tính toán

Ô dưới đây đo ba cách trên cùng giá trị: vòng for trên list, vòng for trên ndarray, phép toán nhân của NumPy. Chuẩn bị đầu vào ngoài phần đo và so kết quả trước khi so tốc độ.


Ô sau dùng làm công cụ đo. Dự đoán thứ tự nhanh/chậm trước khi chạy; không cần học thuộc code đo thời gian.


In [ ]:
from timeit import repeat
from statistics import median

def do_toc_do(n, number=1):
    xs_list = list(range(n))
    xs_np = np.arange(n, dtype=np.int64)
    cac_cach = {
        "for trên list": lambda: [v * 2 for v in xs_list],
        "for trên ndarray": lambda: [v * 2 for v in xs_np],
        "phép toán NumPy": lambda: xs_np * 2,
    }
    ket_qua_chuan = xs_np * 2
    print(f"n={n:,}; mỗi mẫu gồm {number} lần gọi")
    for ten, ham in cac_cach.items():
        np.testing.assert_array_equal(ham(), ket_qua_chuan)  # khởi động và kiểm kết quả
        mau = repeat(ham, repeat=5, number=number)
        print(f"  {ten:20s}: {median(mau)/number*1000:.6f} ms/lần")

# Với mảng nhỏ, đo nhiều lần mỗi mẫu để giảm nhiễu đồng hồ.
do_toc_do(10, number=1000)
do_toc_do(1_000_000)


### Bài tập 2: thay vòng for bằng phép toán mảng

Đoạn code sau tính bình phương chênh lệch của từng cặp số:

```python
x = np.array([10, 12, 11, 20])
y = np.array([12, 11, 15, 20])
ket_qua = []
for u, v in zip(x, y):
    ket_qua.append((v - u) ** 2)
```

1. Tính tay `ket_qua`.
2. Viết lại bằng một biểu thức NumPy, không dùng `for`.
3. Bản viết lại vẫn phải tính trên từng cặp số. Vì sao với mảng lớn, nó thường nhanh hơn?


In [ ]:
# Viết dự đoán và lời giải của bạn ở đây.


**Đối chiếu sau khi đã làm bài.** Chạy ô tham khảo rồi giải thích mọi kết quả khác dự đoán.


In [ ]:
x_bt2 = np.array([10, 12, 11, 20])
y_bt2 = np.array([12, 11, 15, 20])
ket_qua_for = [(v - u) ** 2 for u, v in zip(x_bt2, y_bt2)]
ket_qua_numpy = (y_bt2 - x_bt2) ** 2
print(ket_qua_numpy)
assert ket_qua_numpy.tolist() == ket_qua_for == [4, 1, 16, 0]
# NumPy thực hiện các phép toán bằng mã đã biên dịch,
# giảm xử lý bytecode và đối tượng ở mỗi lượt của vòng for Python.


### Đọc thêm (tuỳ chọn): mảng trung gian

Với ndarray thông thường, `ket_qua = A * 1.1 + 50` gồm hai phép toán:

```python
tam = A * 1.1
ket_qua = tam + 50
```

Phép nhân đọc A và ghi vào `tam`. Phép cộng lại đọc `tam` và ghi vào `ket_qua`. Một dòng công thức vẫn có thể tạo mảng trung gian và duyệt dữ liệu nhiều lần. Khi tối ưu, cần tính cả chi phí tạo mảng và đọc/ghi bộ nhớ.


## 3. Chỉ mục và lát cắt


### Chọn phần tử và lấy lát cắt

`A[hàng, cột]` chọn một ô; chỉ mục bắt đầu từ 0. Mỗi chiều cũng có thể dùng `start:stop:step`, không lấy stop. Bước mặc định là 1.

`A[:2, 1:]` chọn hàng 0, 1 và cột 1, 2. Hãy xác định các giá trị trước khi chạy.


In [ ]:
print(A[2, 1])
print(A[:2, 1:])
assert A[2, 1] == 33
np.testing.assert_array_equal(A[:2, 1:], [[12, 11], [21, 24]])


### Lát cắt tạo một view

`B = A[:2,1:]` có shape `(2,2)`, giá trị `[[12,11],[21,24]]`, offset 8 byte và strides `(24,8)`. View tạo cách nhìn mới trên cùng vùng dữ liệu. Sửa giá trị qua view tác động tới các mảng dùng chung dữ liệu đó.


### Hai tên có thể chỉ cùng một ô

V[0,0] và B[0,1] chỉ cùng một ô dữ liệu. Nếu cần sửa V mà giữ nguyên B, phải thay đổi cách tạo V thế nào?

Đối chiếu: `V = B[:2, 1:].copy()` tạo dữ liệu độc lập.


In [ ]:
B = A.copy()
V = B[:2, 1:]
V[0, 0] = 999
print(B[0])
print(np.shares_memory(B, V))


### Từ chỉ mục đến vị trí trong bộ nhớ

**Strides** cho biết cần dịch bao nhiêu byte khi tăng một chỉ mục lên 1. Với A, `strides = (24, 8)`:

- Tăng chỉ mục hàng: dịch 24 byte (3 số × 8 byte).
- Tăng chỉ mục cột: dịch 8 byte.

Ô `A[2, 1]` là 33, cách đầu vùng dữ liệu của A **2 × 24 + 1 × 8 = 56 byte**.


In [ ]:
print(A.strides)
print(A[2, 1])
print(2 * A.strides[0] + 1 * A.strides[1])


### View có thể bỏ qua các ô

Strides có một giá trị cho mỗi chiều. Với `A[:,::2]`, shape là `(4,2)` và strides là `(24,16)`. Không thể duyệt phẳng mọi phần tử chỉ bằng cách cộng 16 byte liên tục; cả chỉ mục hàng và chỉ mục cột đều tham gia công thức offset.


### Bài tập 3: tìm vị trí phần tử

Mảng A có shape = (4, 3), strides = (24, 8). Dữ liệu được lưu theo thứ tự:

```python
A = np.array([[10, 12, 11],
              [20, 21, 24],
              [30, 33, 31],
              [40, 44, 42]], dtype=np.int64)
```

1. A[1, 2] cách đầu vùng dữ liệu bao nhiêu byte?
2. Phần tử cách đầu vùng dữ liệu 80 byte có giá trị và chỉ mục nào?
3. Tạo B = A.astype(np.int32). Mỗi số còn 4 byte. B.strides bằng bao nhiêu?


In [ ]:
# Viết dự đoán và lời giải của bạn ở đây.


**Đối chiếu sau khi đã làm bài.** Chạy ô tham khảo rồi giải thích mọi kết quả khác dự đoán.


In [ ]:
print("A[1,2]:", A[1,2], "offset:", 1*A.strides[0]+2*A.strides[1])
print("80 byte:", A.ravel()[80//A.itemsize], "chỉ mục:", (3, 1))
print("int32:", A.astype(np.int32).strides)
assert A[1,2] == 24 and A.ravel()[10] == 44


### Bài tập 4: xác định các ô của view

```python
A = np.array([[10, 12, 11],
              [20, 21, 24],
              [30, 33, 31],
              [40, 44, 42]], dtype=np.int64)
V = A[::2, 1:]
```

1. Lát cắt chọn những hàng, cột nào của A? Viết các giá trị của V và V.shape.
2. Tính V.strides. Ô đầu của V cách đầu vùng dữ liệu của A bao nhiêu byte?
3. Nếu gán V[1, 0] = 999, phần tử nào của A thay đổi?


In [ ]:
# Viết dự đoán và lời giải của bạn ở đây.


**Đối chiếu sau khi đã làm bài.** Chạy ô tham khảo rồi giải thích mọi kết quả khác dự đoán.


In [ ]:
B_bt = A.copy()
V_bt = B_bt[::2, 1:]
print(V_bt)
print(V_bt.shape, V_bt.strides)
print("offset ô đầu:", A.itemsize)
V_bt[1,0] = 999
print("Ô đổi:", B_bt[2,1])
assert V_bt.strides == (48,8) and B_bt[2,1] == 999


### Chuyển vị đổi cách đọc dữ liệu

`A.T` là view có shape `(3,4)`, strides `(8,24)`. `reshape` tạo view khi cách đọc cho phép, hoặc phải copy khi cần sắp lại dữ liệu. Không nên khẳng định `reshape` luôn không copy.


In [ ]:
print(A.T.shape, A.T.strides)
print("Chuyển vị dùng chung:", np.shares_memory(A, A.T))
print("Trải A.T theo hàng dùng chung:", np.shares_memory(A, A.T.reshape(-1)))
assert np.shares_memory(A, A.T)
assert not np.shares_memory(A, A.T.reshape(-1))


### Đổi shape có phải chuyển dữ liệu?

A có 12 phần tử. Đổi từ (4, 3) thành (2, 6) vẫn giữ tổng số phần tử. Với A đang liên tục theo hàng, R dùng chung dữ liệu. reshape có thể phải sao chép trong trường hợp khác.

Ở đây `R.strides = (48, 8)`: mỗi hàng có 6 số, mỗi số 8 byte.


In [ ]:
R = A.reshape(2, 6)
print(R)
assert R.shape == (2, 6) and np.shares_memory(A, R)


### Kiểm tra bộ nhớ dùng chung

`np.shares_memory(a, b)` kiểm tra hai mảng có dùng chung phần dữ liệu nào không.


In [ ]:
V = A[:, 1:]
C = A.copy()
print(np.shares_memory(A, V))    # True
print(np.shares_memory(A, A.T))  # True
print(np.shares_memory(A, C))    # False


### Đọc thêm: thuộc tính `.base`

`.base` cho biết đối tượng làm nền cho dữ liệu của mảng. View có thể trỏ tới mảng gốc sâu hơn, nên `b.base is a` trả về `False` chưa đủ để kết luận hai mảng không dùng chung bộ nhớ.


In [ ]:
goc = np.arange(6)
a_view = goc.reshape(2, 3)
b_view = a_view[:, 1:]
print(b_view.base is a_view)                 # False
print(b_view.base is goc)                    # True
print(np.shares_memory(a_view, b_view))      # True


### Lọc phần tử bằng điều kiện

`A % 2 == 0` tạo mask cùng shape với A. `A[mask]` gom các ô True thành mảng một chiều. Nếu mask chỉ chọn hàng, kết quả vẫn giữ các cột.


In [ ]:
mask = (A % 2 == 0)
print(mask)
print(A[mask], A[mask].shape)
print(A[A[:, 0] >= 30])
assert A[mask].tolist() == [10, 12, 20, 24, 30, 40, 44, 42]
assert A[mask].shape == (8,)
assert A[A[:, 0] >= 30].shape == (2, 3)
assert not np.shares_memory(A, A[mask])


### Lọc bằng nhiều điều kiện

Dùng `&` (và), `|` (hoặc); đặt mỗi điều kiện trong ngoặc. Không dùng `and`, `or` để ghép các mask.


In [ ]:
# Từ 20 đến dưới 40
mask = (A >= 20) & (A < 40)
print(A[mask])

# Dưới 12 hoặc từ 40 trở lên
mask = (A < 12) | (A >= 40)
print(A[mask])


### Chọn cặp ô hay chọn cả vùng?

Hai mảng chỉ mục ghép từng cặp. `np.ix_` tạo mọi cặp hàng–cột. Đoán các ô được chọn trước khi chạy.


In [ ]:
print(A[[0, 2], [1, 2]])
print(A[np.ix_([0, 2], [1, 2])])
np.testing.assert_array_equal(A[[0, 2], [1, 2]], [12, 31])
np.testing.assert_array_equal(A[np.ix_([0, 2], [1, 2])], [[12, 11], [33, 31]])


### Lặp và đổi thứ tự chỉ mục

Lấy hàng 1 hai lần; sau đó lấy hàng 0, 2 và cột 2 trước cột 0.


In [ ]:
print(A[[1, 1], :])
print(A[np.ix_([0, 2], [2, 0])])


### Phân biệt view và bản sao

Dự đoán B[0] sau hai phép gán. Vì sao sửa C không tác động tới B?


In [ ]:
B = A.copy()
V = B[:, 1:3]
C = B[:, [1, 2]]
C[0, 0] = -9
V[0, 0] = -1
print(B[0])
print(np.shares_memory(B, V), np.shares_memory(B, C))
np.testing.assert_array_equal(B[0], [10, -1, 11])


### Đọc và gán bằng mask

| Đọc: tạo bản sao | Gán: sửa trực tiếp A |
|---|---|
| `B = A[mask]` | `A[mask] = -1` |
| `A[mask]` đứng một mình hoặc ở vế phải: lấy các giá trị ra mảng mới. | `A[mask]` ở vế trái phép gán: ghi vào các ô được chọn của A. |
| Sửa B không làm đổi A. | Các ô được chọn của A thay đổi. |

Thử với mảng `x_mask` dưới đây để giữ nguyên A cho các phần sau.


In [ ]:
x_mask = np.array([10, 15, 20, 5])
mask = x_mask >= 15
y_mask = x_mask[mask]
y_mask[:] = -1
print(x_mask)         # [10 15 20 5]
x_mask[mask] = -1
print(x_mask)         # [10 -1 -1 5]


Với mảng hai chiều, mask cùng shape chọn từng ô. Dự đoán những ô nào của `B` thay đổi trước khi chạy.


In [ ]:
B = A.copy()
mask = B >= 30
B[mask] = -1
print(B)
print(A)  # A giữ nguyên vì B được tạo bằng A.copy()


### Đọc thêm: gán trực tiếp và lát cắt ngược

Lấy bằng mảng chỉ mục tạo bản sao, nhưng **gán trực tiếp** như `B[:, [1, 2]] = -9` vẫn sửa B.

`A[::-1]` đảo thứ tự hàng bằng stride âm: bắt đầu ở hàng cuối rồi bước ngược. Dự đoán strides và kiểm tra dữ liệu có dùng chung không.

Nguồn: [NumPy: chỉ mục và lát cắt](https://numpy.org/doc/stable/user/basics.indexing.html).


In [ ]:
B = A.copy()
B[:, [1, 2]] = -9
print(B[0])
nguoc = A[::-1]
print(nguoc)
print(nguoc.strides, np.shares_memory(A, nguoc))
assert nguoc.strides == (-24, 8)
assert np.shares_memory(A, nguoc)
np.testing.assert_array_equal(B[0], [10, -9, -9])


## 4. Broadcasting

Cộng theo cột, cộng theo hàng, rồi giải thích bằng shape.


### Cộng vào từng cột

Cộng 1 vào cột 0, cộng 2 vào cột 1, cộng 3 vào cột 2. Ta dùng cùng dãy `b = [1, 2, 3]` cho mọi hàng.


In [ ]:
b = np.array([1, 2, 3], dtype=np.int64)
ket_qua = np.empty_like(A)
for i in range(4):
    for j in range(3):
        ket_qua[i, j] = A[i, j] + b[j]
B_list = ket_qua.tolist()
print(ket_qua)


### Viết phép cộng trên cả mảng

`A + b` cộng dãy `[1, 2, 3]` vào mỗi hàng của A.

**Broadcasting** là quy tắc ghép phần tử khi các mảng có shape khác nhau. Ở đây: `B[i, j] = A[i, j] + b[j]`.


In [ ]:
b = np.array([1, 2, 3], dtype=np.int64)
ket_qua = A + b
print(ket_qua)
np.testing.assert_array_equal(ket_qua, B_list)


### Cộng vào từng hàng

Cộng 10, 20, 30, 40 vào lần lượt các hàng 0, 1, 2, 3.

`d[:, None]` thêm chiều có kích thước 1: `(4,)` thành `(4, 1)`. Mỗi hàng có một số để dùng cho cả ba cột. `None` ở đây tương đương `np.newaxis`.


In [ ]:
d = np.array([10, 20, 30, 40])
d_cot = d[:, None]
print(d_cot)
print(A + d_cot)
assert (A + d_cot).tolist() == [[20,22,21],[40,41,44],[60,63,61],[80,84,82]]


### Quy tắc broadcasting

So sánh shape **từ phải sang trái**. Hai kích thước phải bằng nhau hoặc một bên bằng 1. Chiều thiếu được coi là 1.

| Đầu vào | Chiều trước | Chiều cuối |
|---|---|---|
| A: `(4, 3)` | 4 | 3 |
| b: `(3,)` | thiếu → coi là 1 | 3 |

- Chiều cuối: 3 khớp 3.
- Chiều trước: dùng cùng dãy b cho cả 4 hàng.

Khi so sánh, `(3,)` được xem như `(1, 3)`. **Mảng b vẫn giữ shape `(3,)`**; không cần gọi reshape.

`(4, 3) + (4,)` không hợp lệ: chiều cuối 3 không khớp 4. Muốn cộng một số riêng vào mỗi hàng, dùng mảng có shape `(4, 1)`.


In [ ]:
try:
    A + d
except ValueError as err:
    print("Lỗi dự kiến:", err)
else:
    raise AssertionError("(4,3) và (4,) không thể broadcasting")


### Giữ một cột để cộng theo hàng

Muốn cộng số đầu mỗi hàng vào cả hàng đó, cần shape `(4, 1)`. `A[:, 0]` bỏ chiều cột, còn `A[:, 0:1]` giữ chiều cột. Với shape `(4,)`, chiều cuối 4 không khớp chiều cuối 3 của A.


In [ ]:
print(A[:, 0], A[:, 0].shape)
print(A[:, 0:1].shape)
print(A + A[:, 0:1])
np.testing.assert_array_equal((A + A[:, 0:1])[:, 0], 2 * A[:, 0])


### Bài tập 5: cộng theo hàng và cột

Cho mảng A có 4 hàng, 3 cột. Tạo B qua hai bước:

1. Từ A, cộng lần lượt 1, 2, 3 vào các cột 0, 1, 2.
2. Trên kết quả vừa tính, cộng lần lượt 10, 20, 30, 40 vào các hàng 0, 1, 2, 3.

Ví dụ: `B[0, 1] = A[0, 1] + 2 + 10`.

- Viết code NumPy không dùng `for`. Ghi shape các mảng dùng để cộng.
- Với `A[0] = [10, 12, 11]`, tính `B[0]`.


In [ ]:
# Viết dự đoán và lời giải của bạn ở đây.


**Đối chiếu sau khi đã làm bài.** Chạy ô tham khảo rồi giải thích mọi kết quả khác dự đoán.


In [ ]:
b_bt=np.array([1,2,3])
d_bt=np.array([10,20,30,40])
B_kq=A+b_bt+d_bt[:,None]
print(B_kq)
np.testing.assert_array_equal(B_kq[0],[21,24,24])


## 5. Thống kê và lấy mẫu ngẫu nhiên

Liên hệ project Inside Airbnb. Toàn bộ giá trong phần này là giả lập, đơn vị USD/đêm; không phải kết quả phân tích dữ liệu thật.


### Mô tả mức giá chỗ ở

Giá giả lập USD/đêm: [40,50,60,70,80] và [40,50,60,70,380]. Trong project, dùng số nào để mô tả mức giá điển hình? Một giá cao không tự động là dữ liệu sai.


In [ ]:
for gia_mau in ([40,50,60,70,80], [40,50,60,70,380]):
    print(gia_mau, np.mean(gia_mau), np.median(gia_mau))
assert np.mean([40,50,60,70,380]) == 120


### Giá trong khu vực chênh nhau bao nhiêu?

Giá giả lập USD/đêm: X=[60,60,60,60], Y=[30,30,90,90]. Mean đều 60; std lần lượt 0 và 30 USD. Bình phương độ lệch rồi lấy trung bình là phương sai; lấy căn để trở lại đơn vị USD. NumPy mặc định chia cho n (ddof=0).


In [ ]:
for gia_mau in ([60,60,60,60], [30,30,90,90]):
    print(np.mean(gia_mau), np.var(gia_mau), np.std(gia_mau))
assert np.var([30,30,90,90]) == 900
assert np.std([30,30,90,90]) == 30


### Tính trung bình theo hàng và cột

**axis** chỉ chiều được gộp:

- `A.mean(axis=0)`: tính trung bình từng cột, shape `(3,)`.
- `A.mean(axis=1)`: tính trung bình từng hàng, shape `(4,)`.

Hãy chỉ ra những số tham gia cùng một phép tính trung bình.

Giả sử A là giá của 4 chỗ ở qua 3 kỳ thu thập (USD/đêm, giả lập). Axis 0 gộp các chỗ ở; axis 1 gộp các kỳ của từng chỗ ở.


In [ ]:
print("Trung bình từng cột:", A.mean(axis=0))
print("Trung bình từng hàng:", A.mean(axis=1))


### Trừ trung bình của từng hàng

Bốn số trung bình cần có shape nào để trừ đúng hàng?

`keepdims=True` giữ chiều vừa lấy trung bình, với kích thước 1. Trung bình từng hàng có shape `(4, 1)`, nên có thể trừ trực tiếp khỏi A.

`(4, 3) − (4, 1) → (4, 3)`. Ví dụ hàng đầu `[10, 12, 11]` có trung bình 11; kết quả là `[-1, 1, 0]`.


In [ ]:
tb_hang = A.mean(axis=1, keepdims=True)
chenh_lech = A - tb_hang
print(tb_hang.shape, chenh_lech.shape)
print(chenh_lech.round(2))


### Bài tập 6: sửa phép trừ

Mỗi phần tử cần trừ đi trung bình của hàng chứa nó.

Ví dụ: hàng `[10, 12, 11]` có trung bình 11 → kết quả `[-1, 1, 0]`.

```python
M = np.array([[10, 12, 11],
              [20, 21, 24],
              [30, 33, 31]])
tb = M.mean(axis=1)
chenh_lech = M - tb
```

**Code chạy được nhưng tính sai. Hãy sửa và giải thích lỗi.**


In [ ]:
# Viết dự đoán và lời giải của bạn ở đây.


**Đối chiếu sau khi đã làm bài.** Chạy ô tham khảo rồi giải thích mọi kết quả khác dự đoán.


In [ ]:
M_bt=A[:3]
tb_bt=M_bt.mean(axis=1)
sai_bt=M_bt-tb_bt
dung_bt=M_bt-M_bt.mean(axis=1,keepdims=True)
print("Ô [0,1] sai/đúng:",sai_bt[0,1],dung_bt[0,1])
np.testing.assert_allclose(dung_bt.mean(axis=1),0,rtol=0,atol=1e-12)


### Lấy mẫu để kiểm tra dữ liệu

Chọn chỉ mục ngẫu nhiên để kiểm tra các dòng trong project. Ví dụ dùng 5 giá giả lập USD/đêm. replace=False không chọn trùng một dòng.


In [ ]:
gia = np.array([40,50,60,70,380])
rng = np.random.default_rng(42)
chi_muc = rng.choice(len(gia), size=3, replace=False)
print(gia[chi_muc])
assert len(set(chi_muc)) == 3


### Diễn giải kết quả từ mẫu

Mẫu ngẫu nhiên vẫn có sai số. Seed giúp chạy lại, không bảo đảm tính đại diện. Với project, tính thống kê trên toàn bộ dữ liệu hợp lệ khi có thể; lấy mẫu để đọc và kiểm tra từng dòng.


In [ ]:
print("Toàn bộ:", gia.mean())
for seed in [42,7]:
    chi_muc = np.random.default_rng(seed).choice(len(gia), size=3, replace=False)
    print(seed, gia[chi_muc], gia[chi_muc].mean())


## Tự học: kiểm tra sai số theo độ lớn

Dự đoán rồi thử `np.float32(1e8) + np.float32(1)`. Vì sao cộng 1 mà giá trị có thể không đổi? Hãy liên hệ số bit có nghĩa hữu hạn với việc nhân thêm một lũy thừa lớn của 2. So với float64 và giải thích; không chỉ báo kết quả.


## Đọc thêm: stride bằng 0

Muốn đổi hàng mà vẫn đọc cùng một số, bước theo hàng phải bằng bao nhiêu?

`np.broadcast_to(b, A.shape)` tạo một view có shape `(4, 3)` từ b. Không cần gọi hàm này trước khi viết `A + b`.

View có strides `(0, 8)`. Bước hàng bằng 0: đổi hàng vẫn đọc cùng phần tử của b.

| Ô trong view | Độ lệch (byte) | Giá trị |
|---|---|---|
| `[0, 1]` | 0 × 0 + 1 × 8 = 8 | 2 |
| `[2, 1]` | 2 × 0 + 1 × 8 = 8 | 2 |

View này chỉ đọc. Ba số trong b được dùng chung; phép cộng vẫn cần lưu mảng kết quả.


In [ ]:
b_view = np.broadcast_to(b, A.shape)
print(b_view.shape, b_view.strides)
print(np.shares_memory(b, b_view))


## Tự học với chương 4

Chọn một thao tác trong [chương 4 của McKinney](https://wesmckinney.com/book/numpy-basics): lọc bằng mask, tính thống kê, sắp xếp hoặc mô phỏng. Trình bày:

1. Quan hệ giữa các phần tử trước khi tìm hàm.
2. Shape và dtype đầu vào/đầu ra.
3. Hàm tạo dữ liệu mới hay dùng chung, có sửa đầu vào không.
4. Một ví dụ nhỏ đối chiếu với vòng `for`.

Nguồn về cơ chế: [NumPy internals](https://numpy.org/doc/stable/dev/internals.html), [broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html), [copies/views](https://numpy.org/doc/stable/user/basics.copies.html), [ufunc](https://numpy.org/doc/stable/user/basics.ufuncs.html), [SciPy Lecture Notes](https://scipy-lectures.org/advanced/advanced_numpy/index.html).


In [ ]:
# Kiểm tra ma trận dùng xuyên suốt bài vẫn nguyên vẹn.
np.testing.assert_array_equal(A, [[10,12,11],[20,21,24],[30,33,31],[40,44,42]])
assert A.shape == (4,3) and A.dtype == np.dtype("int64")
assert A.strides == (24,8) and A.nbytes == 96
assert np.broadcast_to(b,A.shape).strides == (0,8)
print("Đã kiểm chứng cấu trúc mảng, broadcasting và dữ liệu gốc.")


## Đọc thêm, không yêu cầu trên lớp: các bit của một số thực

Nếu muốn tìm hiểu cách lưu cụ thể: `6.5 = 110.1₂ = 1.101₂ × 2²`. Chỉ số ₂ chỉ hệ nhị phân. Đưa phần chữ số về dạng `1.…₂` gọi là chuẩn hoá.

Với float32, 32 bit gồm:

- 1 bit dấu: 0 cho số dương.
- 8 bit số mũ: số mũ 2 được lưu thành 2 + 127 = 129. 127 là độ lệch cố định để mã hoá cả số mũ âm.
- 23 bit sau dấu chấm: `101` rồi 20 bit 0. Bit 1 trước dấu chấm được hiểu ngầm với số chuẩn hoá.

![Các bit của 6.5](../img/lecture-03/float32-bits.svg)

Ô dưới dùng `struct` để kiểm chứng; không cần học thuộc API hoặc cách giải mã bit. Các bit được viết từ cao xuống thấp, không phải thứ tự byte trong RAM. Số 0, số dưới chuẩn, vô cực và NaN có cách mã hoá riêng.

Đọc thêm: [Python: sai số số thực](https://docs.python.org/3/tutorial/floatingpoint.html), [IEEE 754: cấu trúc định dạng](https://docs.oracle.com/cd/E19957-01/806-3568/ncg_math.html).


In [ ]:
import struct
bits65 = ''.join(f'{b:08b}' for b in struct.pack('>f', 6.5))
print(bits65[:1], '|', bits65[1:9], '|', bits65[9:])
assert bits65 == '0' + '10000001' + '101' + '0'*20
assert (-1)**int(bits65[0]) * (1 + int(bits65[9:], 2)/2**23) * 2**(int(bits65[1:9],2)-127) == 6.5


## Tra cứu thêm (tự học)

Sau khi đã đặt được câu hỏi, chọn hàm cần dùng: percentile/quantile tìm các mốc của dữ liệu đã sắp; random/normal sinh mẫu từ phân phối; choice lấy mẫu từ mảng. Phần này không trình chiếu trên lớp.


### Phân vị (percentile)

Mốc chia dữ liệu đã sắp xếp theo tỷ lệ phần trăm. percentile dùng 0–100; quantile dùng 0–1. Phân vị 50 là trung vị. Với mẫu hữu hạn, NumPy có thể nội suy giữa các phần tử.


In [ ]:
x_stat = np.array([1,2,3,4,10])
print(np.percentile(x_stat, [25, 50, 75]))
np.testing.assert_array_equal(np.percentile(x_stat, [25, 50, 75]), [2, 3, 4])
np.testing.assert_array_equal(np.quantile(x_stat, [.25, .5, .75]), [2, 3, 4])


### Gieo xúc xắc

default_rng tạo bộ sinh số giả ngẫu nhiên. integers(1,7) sinh số từ 1 đến 6. Khởi tạo lại với cùng seed, giữ cùng môi trường và thứ tự gọi để tái lập ví dụ. Gọi tiếp rng sẽ sinh các số tiếp theo.


In [ ]:
rng = np.random.default_rng(42)
dice = rng.integers(1, 7, size=8)
print(dice)
assert dice.shape == (8,) and np.all((dice >= 1) & (dice <= 6))
np.testing.assert_array_equal(dice, np.random.default_rng(42).integers(1, 7, size=8))


### Sinh số và lấy mẫu

random: phân phối đều trong [0,1). normal(0,1): phân phối chuẩn có trung bình 0, độ lệch chuẩn 1; một mẫu nhỏ không nhất thiết có trung bình 0. choice với replace=False lấy mẫu không hoàn lại.


In [ ]:
uniform_sample = rng.random(size=5)
normal_sample = rng.normal(0, 1, size=5)
chosen = rng.choice([10, 20, 30, 40], size=2, replace=False)
print(uniform_sample)
print(normal_sample)
print(chosen)
assert np.all((uniform_sample >= 0) & (uniform_sample < 1))
assert len(np.unique(chosen)) == 2


---

## Tóm tắt bài học

| Nội dung chính | Vì sao quan trọng |
|---|---|
| Mảng số lưu các phần tử cùng `dtype`; NumPy có mã tính toán đã biên dịch | Hiểu vì sao tính trên mảng thường nhanh hơn vòng `for` Python |
| `shape`, `dtype`, `strides` mô tả mảng | Xác định phần tử nằm ở đâu và cách đọc dữ liệu |
| Lát cắt tạo view; chọn bằng mảng chỉ mục hoặc mask tạo bản sao | Biết khi nào sửa kết quả sẽ làm đổi mảng gốc |
| Broadcasting ghép các chiều từ phải sang trái | Viết đúng phép tính theo hàng và cột |
| Thống kê và lấy mẫu cần gắn với câu hỏi phân tích | Diễn giải giá chỗ ở và tránh kết luận quá mức từ một mẫu nhỏ |

Chạy **Restart & Run all** để kiểm tra notebook từ đầu. Nếu dùng AI, đối chiếu kết quả với phép tính tay hoặc vòng `for` và ghi lại cách kiểm chứng.

**Đọc thêm:** [McKinney, chương 4](https://wesmckinney.com/book/numpy-basics); tài liệu NumPy về [broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html) và [view, bản sao](https://numpy.org/doc/stable/user/basics.copies.html).

**Bài sau:** pandas — làm việc với bảng dữ liệu có tên hàng, tên cột và nhiều kiểu dữ liệu.
